# Section 2: Multiclass Network Intrusion Detection

This fixed experiment uses NSL-KDD to compare a class-balanced Random Forest with a class-weighted 1D CNN across normal, DoS, Probe, R2L, and U2R traffic.

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import json
import random
import shutil
import sys
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
CNN_EPOCHS = 8  # Change to 1 for a quick pipeline check.
COLAB = "google.colab" in sys.modules
ROOT = Path("/content/section_02_workspace") if COLAB else Path.cwd()
DATA_DIR = ROOT / "data/raw/nsl-kdd"
PROCESSED_DIR = ROOT / "data/processed/section_02"
MODELS_DIR = ROOT / "models/section_02"
RESULTS_DIR = ROOT / "reports/section_02"

for directory in (DATA_DIR, PROCESSED_DIR, MODELS_DIR, RESULTS_DIR / "metrics", RESULTS_DIR / "figures"):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Download and load NSL-KDD

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/HoaNP/NSL-KDD-DataSet/master"
for filename in ("KDDTrain+.txt", "KDDTest+.txt"):
    path = DATA_DIR / filename
    if not path.exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{filename.replace('+', '%2B')}", path)

In [ ]:
FEATURE_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login",
    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
]
CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]
NUMERIC_COLUMNS = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
DATA_COLUMNS = [*FEATURE_COLUMNS, "attack_name", "difficulty"]
CLASS_NAMES = ["normal", "dos", "probe", "r2l", "u2r"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
ATTACK_CATEGORY = {
    **{name: "dos" for name in "back land neptune pod smurf teardrop mailbomb apache2 processtable udpstorm".split()},
    **{name: "probe" for name in "ipsweep nmap portsweep satan saint mscan".split()},
    **{name: "r2l" for name in "ftp_write guess_passwd imap multihop phf spy warezclient warezmaster sendmail named snmpgetattack snmpguess xlock xsnoop worm".split()},
    **{name: "u2r" for name in "buffer_overflow loadmodule perl rootkit ps sqlattack xterm httptunnel".split()},
    "normal": "normal",
}

def load_partition(filename):
    frame = pd.read_csv(DATA_DIR / filename, names=DATA_COLUMNS)
    frame["attack_name"] = frame["attack_name"].str.strip().str.lower().str.rstrip(".")
    frame["label_name"] = frame["attack_name"].map(ATTACK_CATEGORY)
    frame["label"] = frame["label_name"].map(CLASS_TO_INDEX).astype(int)
    return frame

full_train = load_partition("KDDTrain+.txt")
test = load_partition("KDDTest+.txt")
train, validation = train_test_split(
    full_train, test_size=0.15, stratify=full_train["label"], random_state=SEED,
)
train, validation = train.reset_index(drop=True), validation.reset_index(drop=True)

profile = pd.DataFrame([
    {"split": name, "records": len(frame), "missing": int(frame.isna().sum().sum()),
     "duplicates": int(frame[DATA_COLUMNS].duplicated().sum()),
     **frame["label_name"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict()}
    for name, frame in (("train", train), ("validation", validation), ("test", test))
])
profile.to_json(PROCESSED_DIR / "data-profile.json", orient="records", indent=2)
display(profile)

## 2. Preprocess features

In [ ]:
columns = ColumnTransformer([
    ("numeric", StandardScaler(), NUMERIC_COLUMNS),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLUMNS),
], verbose_feature_names_out=False)
feature_pipeline = Pipeline([
    ("columns", columns),
    ("variance", VarianceThreshold()),
    ("selection", SelectKBest(f_classif, k=64)),
])

train_features = feature_pipeline.fit_transform(train[FEATURE_COLUMNS], train["label"]).astype(np.float32)
validation_features = feature_pipeline.transform(validation[FEATURE_COLUMNS]).astype(np.float32)
test_features = feature_pipeline.transform(test[FEATURE_COLUMNS]).astype(np.float32)
train_labels = train["label"].to_numpy(np.int64)
validation_labels = validation["label"].to_numpy(np.int64)
test_labels = test["label"].to_numpy(np.int64)

feature_names = feature_pipeline.named_steps["columns"].get_feature_names_out()
feature_names = feature_names[feature_pipeline.named_steps["variance"].get_support()]
feature_names = feature_names[feature_pipeline.named_steps["selection"].get_support()].tolist()
(PROCESSED_DIR / "selected-features.json").write_text(json.dumps(feature_names, indent=2))
display(pd.DataFrame({"split": ["train", "validation", "test"],
                      "records": [len(train_features), len(validation_features), len(test_features)],
                      "features": [train_features.shape[1]] * 3}))

In [ ]:
def evaluate(name, labels, probabilities, filename):
    predictions = probabilities.argmax(axis=1)
    binary_labels = label_binarize(labels, classes=np.arange(len(CLASS_NAMES)))
    metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": precision_score(labels, predictions, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, predictions, average="macro", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
        "macro_average_precision": average_precision_score(binary_labels, probabilities, average="macro"),
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
        "classification_report": classification_report(
            labels, predictions, labels=np.arange(len(CLASS_NAMES)),
            target_names=CLASS_NAMES, output_dict=True, zero_division=0,
        ),
    }
    (RESULTS_DIR / "metrics" / f"{filename}.json").write_text(json.dumps(metrics, indent=2))

    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay.from_predictions(
        labels, predictions, display_labels=CLASS_NAMES, cmap="Blues", colorbar=False, ax=axes[0],
    )
    axes[0].set_title("Confusion matrix")
    for index, class_name in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(binary_labels[:, index], probabilities[:, index])
        axes[1].plot(recall, precision, label=class_name)
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curves")
    axes[1].legend()
    figure.suptitle(name)
    figure.tight_layout()
    figure.savefig(RESULTS_DIR / "figures" / f"{filename}-evaluation.png", dpi=180)
    plt.show()
    return metrics

## 3. Random Forest

In [ ]:
forest = RandomForestClassifier(
    n_estimators=180, max_depth=28, max_features="sqrt",
    class_weight="balanced_subsample", n_jobs=1, random_state=SEED,
)
forest.fit(train_features, train_labels)
forest_probabilities = forest.predict_proba(test_features)
forest_metrics = evaluate("Random Forest", test_labels, forest_probabilities, "random-forest")
joblib.dump({"feature_pipeline": feature_pipeline, "model": forest, "class_names": CLASS_NAMES},
            MODELS_DIR / "random-forest.joblib")
display(pd.DataFrame(forest_metrics["classification_report"]).T)

## 4. Class-weighted 1D CNN

In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 5),
        )

    def forward(self, values):
        return self.network(values.unsqueeze(1))

def make_loader(features, labels, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))
    return DataLoader(dataset, batch_size=512, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(SEED) if shuffle else None)

train_loader = make_loader(train_features, train_labels, True)
validation_loader = make_loader(validation_features, validation_labels)
test_loader = make_loader(test_features, test_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = FeatureCNN().to(device)
counts = np.bincount(train_labels, minlength=5)
class_weights = np.sqrt(len(train_labels) / (5 * counts))
class_weights = class_weights / class_weights.mean()
loss_function = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001, weight_decay=0.0001)

def cnn_predictions(loader):
    cnn.eval()
    probabilities = []
    with torch.no_grad():
        for values, _ in loader:
            probabilities.append(torch.softmax(cnn(values.to(device)), dim=1).cpu().numpy())
    return np.concatenate(probabilities)

In [ ]:
history = []
best_validation_f1 = -1
for epoch in range(1, CNN_EPOCHS + 1):
    cnn.train()
    total_loss = 0
    for values, labels in train_loader:
        values, labels = values.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_function(cnn(values), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    validation_probabilities = cnn_predictions(validation_loader)
    validation_f1 = f1_score(validation_labels, validation_probabilities.argmax(1), average="macro")
    history.append({"epoch": epoch, "training_loss": total_loss / len(train), "validation_macro_f1": validation_f1})
    if validation_f1 > best_validation_f1:
        best_validation_f1 = validation_f1
        best_state = {name: value.detach().cpu().clone() for name, value in cnn.state_dict().items()}

cnn.load_state_dict(best_state)
history_frame = pd.DataFrame(history)
display(history_frame)
history_frame.set_index("epoch").plot(subplots=True, figsize=(7, 5), title=["Training loss", "Validation macro F1"])
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures/cnn-training-history.png", dpi=180)
plt.show()

cnn_probabilities = cnn_predictions(test_loader)
cnn_metrics = evaluate("1D CNN", test_labels, cnn_probabilities, "cnn")
cnn_metrics.update({"best_validation_macro_f1": best_validation_f1, "epochs": CNN_EPOCHS, "device": str(device)})
(RESULTS_DIR / "metrics/cnn.json").write_text(json.dumps(cnn_metrics, indent=2))
torch.save({"model_state": best_state, "class_names": CLASS_NAMES}, MODELS_DIR / "cnn.pt")
display(pd.DataFrame(cnn_metrics["classification_report"]).T)

## 5. Compare and export

In [ ]:
metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "macro_average_precision"]
comparison = pd.DataFrame([
    {"model": "Random Forest", **{metric: forest_metrics[metric] for metric in metric_names}},
    {"model": "1D CNN", **{metric: cnn_metrics[metric] for metric in metric_names}},
])
comparison.to_csv(RESULTS_DIR / "model-comparison.csv", index=False)
(RESULTS_DIR / "run-summary.json").write_text(json.dumps({
    "train_records": len(train), "validation_records": len(validation),
    "test_records": len(test), "selected_features": train_features.shape[1],
    "cnn_epochs": CNN_EPOCHS,
}, indent=2))
display(comparison.style.format({metric: "{:.4f}" for metric in metric_names}).highlight_max(subset=metric_names, color="#d9ead3"))

if COLAB:
    export_dir = ROOT / "section_02_export"
    shutil.copytree(RESULTS_DIR, export_dir / "reports", dirs_exist_ok=True)
    shutil.copytree(MODELS_DIR, export_dir / "models", dirs_exist_ok=True)
    shutil.make_archive("/content/section_02_results", "zip", root_dir=export_dir)

## Interpretation

- Use macro metrics and per-class results because R2L and U2R are rare.
- The official test set contains attack types absent from training, so it is harder than a random split.
- Compare minority-class recall as well as overall accuracy.
- NSL-KDD is an old benchmark and does not establish performance on modern network traffic.